# Group Classification Inference (Unified Config Pipeline)

This notebook runs the unified `inference_pipeline` using a config-only setup.


## 1) Environment Setup


In [1]:
from pathlib import Path

from pioneerml.integration.zenml import load_step_output
from pioneerml.integration.zenml import utils as zenml_utils
from pioneerml.pipeline.pipelines.inference import inference_pipeline

PROJECT_ROOT = zenml_utils.find_project_root()
zenml_utils.setup_zenml_for_notebook(root_path=PROJECT_ROOT, use_in_memory=True)


Using ZenML repository root: /workspace
Ensure this is the top-level of your repo (.zen must live here).


## 2) Select Input Files and Output Location


In [2]:
# Input parquet files
_data_dir = Path(PROJECT_ROOT) / "data"
parquet_paths = sorted(_data_dir.glob("ml_output_*.parquet"))

# Optional: use a subset for quick checks
parquet_paths = parquet_paths[:1]

parquet_paths = [str(p.resolve()) for p in parquet_paths]
if not parquet_paths:
    raise RuntimeError(f"No parquet files found in {_data_dir}")

# Model path override (None => auto-resolve latest in trained_models/<model_subdir>)
model_path = None

# Output location
output_dir = str((Path(PROJECT_ROOT) / "artifacts" / "inference_small" / "group_classifier").resolve())


## 3) Reusable Config Helpers


In [3]:
from pioneerml.plugin.runtime import ensure_plugins_loaded
ensure_plugins_loaded()

from pioneerml_base_plugin.group_classifier.pipeline import load_config
from pioneerml_base_plugin.utils.config_loader import with_loader_sources, with_model_handle_path, with_writer_output


## 4) Build Step Config Blocks


In [4]:
pipeline_config = load_config()["inference"]
pipeline_config = with_loader_sources(
    pipeline_config,
    main_sources=parquet_paths,
    optional_sources_by_name={},
)
pipeline_config = with_model_handle_path(pipeline_config, model_path=model_path)
pipeline_config = with_writer_output(
    pipeline_config,
    output_dir=output_dir,
)


## 5) Assemble `pipeline_config` and Run


In [5]:
run = inference_pipeline.with_options(enable_cache=False)(
    pipeline_config=pipeline_config,
)


Initiating a new run for the pipeline: inference_pipeline.
Caching is disabled by default for inference_pipeline.
Using user: default
Using stack: default
  deployer: default
  artifact_store: default
  orchestrator: default
You can visualize your pipeline runs in the ZenML Dashboard. In order to try it locally, please run zenml login --local.
Step build_model_handle has started.
[build_model_handle] No materializer is registered for type <class 'pioneerml.integration.pytorch.model_handles.torchscript_model_handle.TorchScriptModelHandle'>, so the default Pickle materializer was used. Pickle is not production ready and should only be used for prototyping as the artifacts cannot be loaded when running with a different Python version. Please consider implementing a custom materializer for type <class 'pioneerml.integration.pytorch.model_handles.torchscript_model_handle.TorchScriptModelHandle'> according to the instructions at https://docs.zenml.io/concepts/artifacts/materializers
Step bui

## 6) Inspect Inference Outputs


In [6]:
inference_output = load_step_output(run, "run_inference")
print("inference_output:", inference_output)

predictions_paths = [Path(p) for p in (inference_output.get("predictions_paths") or [])]
if not predictions_paths and inference_output.get("predictions_path"):
    predictions_paths = [Path(inference_output["predictions_path"])]

print("predictions_paths:")
for p in predictions_paths:
    print(" ", p)


inference_output: {'predictions_path': '/workspace/artifacts/inference_small/group_classifier/ml_output_000_preds.parquet', 'predictions_paths': ['/workspace/artifacts/inference_small/group_classifier/ml_output_000_preds.parquet'], 'timestamped_predictions_path': '/workspace/artifacts/inference_small/group_classifier/ml_output_000_preds_20260326_011825.parquet', 'timestamped_predictions_paths': ['/workspace/artifacts/inference_small/group_classifier/ml_output_000_preds_20260326_011825.parquet']}
predictions_paths:
  /workspace/artifacts/inference_small/group_classifier/ml_output_000_preds.parquet


## 7) Optional: Quick Parquet Validation


In [7]:
import gc

import pyarrow as pa
import pyarrow.parquet as pq

if not predictions_paths:
    raise RuntimeError("No prediction parquet files were exported.")

pf = pq.ParquetFile(predictions_paths[0])
print("file:", predictions_paths[0])
print("rows:", pf.metadata.num_rows)
print(pf.schema_arrow)

if pf.num_row_groups > 0:
    sample = pf.read_row_group(0).slice(0, 3)
    print(sample)
else:
    sample = None
    print("No row groups found.")

del sample, pf
gc.collect()
pa.default_memory_pool().release_unused()


file: /workspace/artifacts/inference_small/group_classifier/ml_output_000_preds.parquet
rows: 1024
event_id: int64
pred_pion: list<element: float>
  child 0, element: float
pred_muon: list<element: float>
  child 0, element: float
pred_mip: list<element: float>
  child 0, element: float
time_group_ids: list<element: int64>
  child 0, element: int64
pyarrow.Table
event_id: int64
pred_pion: list<element: float>
  child 0, element: float
pred_muon: list<element: float>
  child 0, element: float
pred_mip: list<element: float>
  child 0, element: float
time_group_ids: list<element: int64>
  child 0, element: int64
----
event_id: [[0,1,2]]
pred_pion: [[[0.9994904,2.9399277e-10,2.1876029e-7],[0.9995708,5.1108943e-11],[0.99999404,6.130237e-13,0.9997832,0.0011467678,0.0000025945374]]]
pred_muon: [[[0.034318868,1,0.00003651348],[0.030226598,1],[0.0060750633,1,0.027705772,0.9971384,0.00009613148]]]
pred_mip: [[[0.000046869758,1.5495972e-9,0.99999666],[0.00003375647,9.859121e-11],[3.8295693e-7,7.7